# DeBERTa YouTube Sentiment Analysis + Flask API

This notebook downloads the YouTube Comments dataset from Kaggle, fine-tunes a DeBERTa model for 3-class sentiment (positive, negative, neutral), and then provides a simple Flask API to serve predictions. You can adapt or split cells to your liking.

In [10]:
!pip -q install kagglehub

In [11]:
!pip install --upgrade transformers


  Using cached transformers-4.50.3-py3-none-any.whl.metadata (39 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
Using cached transformers-4.50.3-py3-none-any.whl (10.2 MB)
Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl (2.4 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.13.3
    Uninstalling tokenizers-0.13.3:
      Successfully uninstalled tokenizers-0.13.3
  Attempting uninstall: transformers
    Found existing installation: transformers 4.28.0
    Uninstalling transformers-4.28.0:
      Successfully uninstalled transformers-4.28.0


In [12]:
!pip install --upgrade --force-reinstall accelerate


  Using cached accelerate-1.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached psutil-7.0.0-cp37-abi3-win_amd64.whl.metadata (23 kB)
  Using cached PyYAML-6.0.2-cp310-cp310-win_amd64.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.30.1-py3-none-any.whl.metadata (13 kB)
  Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.3.2-py3-none-any.whl.metadata (11 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.13.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.1.0+cpu requires torch==2.1.0, but you have torch 2.6.0 which is incompatible.
torchvision 0.16.0+cpu requires torch==2.1.0, but you have torch 2.6.0 which is incompatible.


In [13]:
!pip install "accelerate>=0.26.0"


In [14]:
!pip install "transformers[torch]"


In [15]:
!pip -q install torch scikit-learn pandas

In [16]:
!pip install "accelerate>=0.26.0"


In [17]:
!pip -q install kagglehub

In [18]:
!pip install transformers datasets

In [19]:
!pip -q install Flask Werkzeug

In [20]:
###########################################################
# 1. IMPORTS
###########################################################
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

import torch
import torch.nn.functional as F

# Hugging Face Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

# For Kaggle dataset
import kagglehub

# Flask for API
from flask import Flask, request, jsonify

###########################################################
# 2. DOWNLOAD DATASET FROM KAGGLE
###########################################################
print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("atifaliak/youtube-comments-dataset")
print("Dataset downloaded to:", path)

# The path is typically a local folder. Let's find the .csv file
csv_file = os.path.join(path, "YoutubeCommentsDataSet.csv")

###########################################################
# 3. LOAD & PREPARE DATA
###########################################################
df = pd.read_csv(csv_file)

print("Sample of raw data:")
print(df.head())

# We'll assume columns: "Comment" and "Sentiment" in ["positive", "negative", "neutral"].
df = df[df["Sentiment"].isin(["positive", "negative", "neutral"])]

# Encode labels
label_map = {"positive": 0, "negative": 1, "neutral": 2}
df["label"] = df["Sentiment"].apply(lambda x: label_map[x])

# Rename "Comment" column to "text"
df.rename(columns={"Comment": "text"}, inplace=True)

# Clean up text (remove NaN or empty)
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)
df = df[df["text"].str.strip() != ""]

# Split train/test (80/20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\ProgramData\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
###########################################################
# 4. HUGGING FACE DATASET & TOKENIZATION
###########################################################

model_name = "microsoft/deberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class YTCommentsDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encodings = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encodings.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

train_dataset = YTCommentsDataset(train_df, tokenizer)
test_dataset = YTCommentsDataset(test_df, tokenizer)

###########################################################
# 5. MODEL INIT & TRAINING
###########################################################

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./deberta_results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=1,  # Increase for better performance
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=50,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting fine-tuning of DeBERTa...")
trainer.train()
print("Training complete.")

# Evaluate final model on test set
metrics = trainer.evaluate(test_dataset)
print("Test set metrics:", metrics)

# Print classification report
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["positive","negative","neutral"]))

# Save the final model
trainer.save_model("./deberta_finetuned")
tokenizer.save_pretrained("./deberta_finetuned")

In [ ]:
###########################################################
# 6. BUILD A FLASK API TO SERVE PREDICTIONS
###########################################################
app = Flask(__name__)

# Load model and tokenizer for inference
inference_model = AutoModelForSequenceClassification.from_pretrained("./deberta_finetuned")
inference_tokenizer = AutoTokenizer.from_pretrained("./deberta_finetuned")
inference_model.eval()

def predict_sentiment(texts):
    """Predict sentiment for a list of input strings, returning pos/neg/neu probabilities."""
    inputs = inference_tokenizer(
        texts,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = inference_model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
    # Each row in `probs` is [prob_pos, prob_neg, prob_neu]
    # Our label mapping: positive=0, negative=1, neutral=2.
    return probs

@app.route("/predict", methods=["POST"])
def predict():
    """Expects a JSON payload with 'texts' (array) or 'text' (string)."""
    data = request.get_json(force=True)
    if "texts" in data:
        texts = data["texts"]
    elif "text" in data:
        texts = [data["text"]]
    else:
        return jsonify({"error": "No input text(s) provided"}), 400

    # Get probabilities
    probs = predict_sentiment(texts)
    # Convert model outputs to label & confidence
    label_map_inv = {0: "positive", 1: "negative", 2: "neutral"}
    results = []
    for p in probs:
        max_idx = np.argmax(p)
        label = label_map_inv[max_idx]
        confidence = float(p[max_idx])
        results.append({
            "label": label,
            "confidence": confidence,
            "all_probs": {
                "positive": float(p[0]),
                "negative": float(p[1]),
                "neutral": float(p[2]),
            }
        })

    if len(texts) == 1:
        return jsonify(results[0])
    else:
        return jsonify(results)

if __name__ == "__main__":
    # Running the Flask API within Colab or local Jupyter
    # If you're in Colab, you need ngrok/localtunnel to expose port.
    print("Starting Flask on port 5000...")
    app.run(host="0.0.0.0", port=5000, debug=False)
    print("Flask app has stopped.")